<a href="https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


**Finding 1: The Content Archetypes (k-means, k=5, exploratory appendix)**

The paper clusters 61.8K active-content pieces into 5 segments using k-means, but reports that the "names in the original analysis were heuristic and sometimes duplicated", with three of the five clusters sharing the identical internal label "Rising Stars."

Where does the label come from? There's no label here — this is unsupervised, same as my own lane, so the question is about what the clusters actually represent. The paper is honest that the naming was informal, which is a good practice I want to hold myself to.

Does the validation design carry the claim? The paper doesn't report a silhouette score or any other internal validation metric for the k=5 choice — it's unclear whether k=5 was chosen by a systematic sweep (like I did) or a default/assumed value. A constructive methodology question: how was k=5 selected, and would a different k produce meaningfully different or more separable segments? Since three of five clusters share a heuristic label, that itself hints the chosen k may be finer than the data naturally separates into — a lower k, or a documented silhouette comparison, would strengthen the claim that these are five distinct archetypes rather than three-plus-noise.

**Finding 2: Random Forest feature importance for Health Score**

The paper reports Average Position (43%) and Impressions (32%) as the top predictors of Health Score, but is transparent that "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal."

Where does the label come from? The label (Health Score) is explicitly a defined rule, not an observed outcome — it's a weighted formula built from impressions, position, CTR, and scroll depth. This is exactly the "target must be observed, not defined" caution from the framing-ml-problems skill: since two of the model's top three predictors (position, impressions) are literal ingredients of the score being predicted, the high importance is close to circular by construction.

Does the validation design carry the claim? The paper reports an 80/20 holdout split, which is good practice, but a holdout split doesn't fix a target-leakage problem — the model can still score well on held-out data by re-deriving inputs that are already part of the formula it's predicting. A constructive methodology question: would feature importance look meaningfully different if position and impressions (both Health Score components) were excluded, isolating which non-formula signals — like scroll depth or CTR — actually add predictive value? The paper's own caveat already flags this, which I think is the right instinct — I'd just push it one step further into an ablation test.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Rebuild content_level (self-contained, same as w05)

# Setup
import pandas as pd
import numpy as np
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

fact_cols = ["report_date","client_hash_id","content_hash_id","gsc_data_available","ga4_data_available",
             "gsc_impressions","gsc_clicks","gsc_avg_position","ga4_pageviews","ga4_sessions",
             "ga4_engaged_sessions","ga4_total_engagement_sec"]
panel_daily = pd.read_parquet(
    f"{HF_BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    columns=fact_cols, storage_options={"token": HF_TOKEN}
)
dim_cols = ["client_hash_id","content_hash_id","content_type"]
dim_content = pd.read_parquet(f"{HF_BASE}/dim_content.parquet", columns=dim_cols, storage_options={"token": HF_TOKEN})

for col in ["client_hash_id","content_hash_id"]:
    panel_daily[col] = panel_daily[col].astype("category")
    dim_content[col] = dim_content[col].astype("category")

dim_content_clean = dim_content.drop_duplicates(subset=["client_hash_id","content_hash_id"], keep="first")
panel_daily = panel_daily.merge(dim_content_clean, on=["client_hash_id","content_hash_id"], how="left")

content_level = panel_daily.groupby(["client_hash_id","content_hash_id"], observed=True).agg(
    gsc_impressions=("gsc_impressions","sum"),
    gsc_clicks=("gsc_clicks","sum"),
    gsc_avg_position=("gsc_avg_position","mean"),
    ga4_engaged_sessions=("ga4_engaged_sessions","sum"),
    content_type=("content_type","first"),
).reset_index()
content_level["ctr"] = content_level["gsc_clicks"] / content_level["gsc_impressions"].replace(0, np.nan)

feature_cols = ["gsc_impressions","gsc_clicks","gsc_avg_position","ga4_engaged_sessions"]
print("Shape:", content_level.shape)

Shape: (331437, 8)


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GroupShuffleSplit

rng = np.random.default_rng(42)

# BEFORE: naive split (ignores client grouping — random row split)
X_all = StandardScaler().fit_transform(content_level[feature_cols].fillna(0))
train_idx_naive = rng.choice(len(X_all), size=int(0.7*len(X_all)), replace=False)
test_idx_naive = np.setdiff1d(np.arange(len(X_all)), train_idx_naive)

km_naive = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_all[train_idx_naive])
naive_test_labels = km_naive.predict(X_all[test_idx_naive])
idx_s = rng.choice(len(test_idx_naive), size=min(20000, len(test_idx_naive)), replace=False)
naive_silhouette = silhouette_score(X_all[test_idx_naive][idx_s], naive_test_labels[idx_s])
print("BEFORE (naive random split) — held-out silhouette:", round(naive_silhouette, 3))

# AFTER: honest grouped split by client (same as w05)
groups = content_level["client_hash_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(content_level, groups=groups))

X_train = StandardScaler().fit(content_level.iloc[train_idx][feature_cols].fillna(0))
scaler = StandardScaler().fit(content_level.iloc[train_idx][feature_cols].fillna(0))
X_train_t = scaler.transform(content_level.iloc[train_idx][feature_cols].fillna(0))
X_test_t = scaler.transform(content_level.iloc[test_idx][feature_cols].fillna(0))

km_grouped = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_train_t)
grouped_test_labels = km_grouped.predict(X_test_t)
idx_s2 = rng.choice(len(X_test_t), size=min(20000, len(X_test_t)), replace=False)
grouped_silhouette = silhouette_score(X_test_t[idx_s2], grouped_test_labels[idx_s2])
print("AFTER (grouped-by-client split) — held-out silhouette:", round(grouped_silhouette, 3))

print(f"\nDifference: {grouped_silhouette - naive_silhouette:.3f}")

BEFORE (naive random split) — held-out silhouette: 0.911
AFTER (grouped-by-client split) — held-out silhouette: 0.907

Difference: -0.004


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Re-run the leakage hunt from w03/ML-05, on the final feature set used here
final_features = feature_cols
future_terms = ["future","next_month","is_published","is_deleted","optimized_date","optimization_eligible"]
leaked = [c for c in final_features if any(t in c.lower() for t in future_terms)]
print("Final features:", final_features)
print("Flagged as leaked:", leaked)
assert len(leaked) == 0, "Leakage detected!"

# Confirm no April/June data anywhere in this session's variables
print("panel_daily date range:", panel_daily["report_date"].min(), "-", panel_daily["report_date"].max())
assert panel_daily["report_date"].max() < pd.Timestamp("2026-04-01").date(), "Data bleeds past March!"
print("Leakage audit passed: final feature set is same-window, non-flag, observed-only.")

Final features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_engaged_sessions']
Flagged as leaked: []
panel_daily date range: 2026-03-01 - 2026-03-31
Leakage audit passed: final feature set is same-window, non-flag, observed-only.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In this March 2026 sample, content in Cluster 1 is observed to have substantially higher impressions, clicks, and engagement than Cluster 0 — a directional, decision-support signal that this content is worth prioritizing for protection. This is descriptive, not causal: it does not explain why this content performs better, and — as the FlyRank paper's own archetype section shows (three of its five clusters shared an identical heuristic label) — cluster boundaries found by k-means should be treated as a starting point for review, not a fixed or definitive segmentation."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.